In [ ]:
# Side-by-side display (no subplots, no saving to files):
# Render two separate figures and place them next to each other using HTML.
# Left:  Acc@5 vs FID  (marker size by GFLOPS bins)
# Right: GFLOPS vs FID (marker size by Acc@5 bins)
# Color: #FF2D55, marker: 'o'

import io, base64
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import HTML, display

# ----- Data -----
rows = [
  ("vit_h_14",          98.694, 1016.72, 11.04),
  ("regnet_y_128gf",    97.844, 127.52,  13.17),
  ("regnet_y_16gf_v1",  97.244, 15.91,   10.26),
  ("convnext_base",     96.870, 15.36,   10.64),
  ("efficientnet_b5",   96.628, 10.27,   11.53),
  ("regnet_y_16gf_v2",  96.328, 15.91,    9.64),
  ("swin_v2_t",         96.132, 5.94,     9.73),
  ("swin_t",            95.776, 4.49,     9.65),
  ("regnet_y_32gf",     95.340, 32.28,    None),
  ("regnet_y_8gf",      95.048, 8.47,     9.54),
  ("regnet_y_3_2gf",    94.576, 3.18,     9.50),
  ("resnet152",         94.046, 11.51,    9.49),
  ("resnet101",         93.546, 7.80,     9.59),
  ("regnet_y_800mf",    93.136, 0.83,     9.40),
  ("mobilenet_v3_large",92.566, 0.22,     None),
  ("regnet_y_400mf",    91.716, 0.40,     None),
  ("regnet_x_400mf",    90.950, 0.41,     None),
  ("mobilenet_v2",      90.286, 0.30,     None),
  ("shufflenet_v2_x1_0",88.316, 0.14,     None),
  ("alexnet",           79.066, 0.71,     None),
]

# Keep rows with FID present
rows = [r for r in rows if r[3] is not None]
names   = np.array([r[0] for r in rows])
acc5    = np.array([r[1] for r in rows], dtype=float)
gflops  = np.array([r[2] for r in rows], dtype=float)
fid     = np.array([r[3] for r in rows], dtype=float)

COLOR = "#FF2D55"
size_levels = np.array([40, 80, 130, 190, 260], dtype=float)

# ---------- Figure A: Acc@5 vs FID (size by GFLOPS bins) ----------
n_bins = 5
logg = np.log10(gflops)
edges_log = np.quantile(logg, np.linspace(0, 1, n_bins + 1))
edges_log = np.unique(edges_log)
if edges_log.size < n_bins + 1:
    edges_log = np.log10(np.array([0.3, 1, 3, 10, 30, 100, 300, 3000]))
edges_g = np.power(10.0, edges_log)
edges_g[0]  = min(edges_g[0], gflops.min()*0.999)
edges_g[-1] = max(edges_g[-1], gflops.max()*1.001)
if edges_g.size > n_bins + 1:
    idx = np.linspace(0, edges_g.size-1, n_bins+1).round().astype(int)
    edges_g = edges_g[idx]

bin_idx_g = np.digitize(gflops, edges_g) - 1
bin_idx_g = np.clip(bin_idx_g, 0, len(size_levels)-1)
sizes_A = size_levels[bin_idx_g]

figA = plt.figure(figsize=(6.4,4.8))
plt.scatter(acc5, fid, s=sizes_A, c=COLOR, marker='o', alpha=0.9, edgecolors='none')
plt.xlabel("Acc@5 (%)")
plt.ylabel("FID (lower is better)")
plt.title("Acc@5 vs FID  (size by GFLOPS bins)")
plt.grid(True, alpha=0.25)
handles = []
for b in range(len(size_levels)):
    lo = edges_g[b]
    hi = edges_g[b+1] if b+1 < len(edges_g) else None
    label = f"{lo:.2g}–{hi:.2g} GFLOPS" if hi is not None else f">{lo:.2g} GFLOPS"
    handles.append(Line2D([0],[0], marker='o', color='w',
                          label=label, markerfacecolor=COLOR, markeredgecolor='none',
                          markersize=(size_levels[b]**0.5)))
plt.legend(handles=handles, title="GFLOPS bins (size)", loc='best', framealpha=0.9)
plt.tight_layout()

bufA = io.BytesIO()
figA.savefig(bufA, format="png", dpi=200, bbox_inches="tight")
plt.close(figA)
b64A = base64.b64encode(bufA.getvalue()).decode("ascii")

# ---------- Figure B: GFLOPS vs FID (size by Acc@5 bins) ----------
edges_a = np.quantile(acc5, np.linspace(0, 1, n_bins + 1))
edges_a = np.unique(edges_a)
if edges_a.size < n_bins + 1:
    edges_a = np.linspace(float(acc5.min()), float(acc5.max()), n_bins + 1)

bin_idx_a = np.digitize(acc5, edges_a) - 1
bin_idx_a = np.clip(bin_idx_a, 0, len(size_levels)-1)
sizes_B = size_levels[bin_idx_a]

figB = plt.figure(figsize=(6.4,4.8))
plt.scatter(gflops, fid, s=sizes_B, c=COLOR, marker='o', alpha=0.9, edgecolors='none')
plt.xscale("log")
plt.xlabel("GFLOPS (log scale)")
plt.ylabel("FID (lower is better)")
plt.title("GFLOPS vs FID  (size by Acc@5 bins)")
plt.grid(True, which="both", alpha=0.25)
handles2 = []
for b in range(len(size_levels)):
    lo = edges_a[b]
    hi = edges_a[b+1] if b+1 < len(edges_a) else None
    label = f"{lo:.2f}–{hi:.2f}% Acc@5" if hi is not None else f">{lo:.2f}% Acc@5"
    handles2.append(Line2D([0],[0], marker='o', color='w',
                           label=label, markerfacecolor=COLOR, markeredgecolor='none',
                           markersize=(size_levels[b]**0.5)))
plt.legend(handles=handles2, title="Acc@5 bins (size)", loc='best', framealpha=0.9)
plt.tight_layout()

bufB = io.BytesIO()
figB.savefig(bufB, format="png", dpi=200, bbox_inches="tight")
plt.close(figB)
b64B = base64.b64encode(bufB.getvalue()).decode("ascii")

# ---------- Side-by-side HTML ----------
html = f"""
<div style="display:flex; gap:12px; align-items:flex-start;">
  <img src="data:image/png;base64,{b64A}" style="width:50%; height:auto;" />
  <img src="data:image/png;base64,{b64B}" style="width:50%; height:auto;" />
</div>
"""
display(HTML(html))
